# 01 Data Preprocessing

## Overview

This notebook runs the full data preprocessing pipeline:
- **Document parsing**: PDF, DOCX, TXT (auto encoding detection)
- **Text cleaning**: Remove headers, footers, page numbers, structural noise
- **Encoding repair**: Auto-detect and fix garbled text
- **Traditional → Simplified Chinese**: Automatic conversion
- **Deduplication**: Line-, paragraph-, and chunk-level
- **Quality gates**: Chinese ratio check, valid character validation
- **Checkpointing**: Resume from interruption, avoid data loss

## Outputs

- `data/processed_corpus/chunks_clean.jsonl` – final chunks file
- `data/processed_corpus/audit_log.jsonl` – audit log (optional)
- `data/processed_corpus/checkpoint.json` – checkpoint (removed when done)


## 1. Environment setup

In [1]:
# 1. Environment setup
import sys
from pathlib import Path
import logging
import multiprocessing

# Detect project root
notebook_dir = Path().resolve()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    current = notebook_dir
    while current != current.parent:
        if (current / "configs").exists():
            project_root = current
            break
        current = current.parent
    else:
        project_root = notebook_dir.parent

sys.path.insert(0, str(project_root))
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
from src.data_processing.data_preprocessor_01_01 import DataPreprocessor

print("Environment ready.")
print(f"Project root: {project_root}")


✅ 环境初始化完成
📁 项目根目录: D:\9_Projects\Veritas\VeritasCarbon_VLDB


## 2. Initialize preprocessor

In [2]:
# 2. Initialize preprocessor
preprocessor = DataPreprocessor(
    config_path="configs/config.yaml",
    project_root=project_root
)
print("Preprocessor initialized.")
print("Config:")
for key, value in preprocessor.config['data'].items():
    if isinstance(value, bool):
        print(f"  - {key}: {value}")
    elif isinstance(value, (int, float)):
        print(f"  - {key}: {value}")
    else:
        print(f"  - {key}: {value}")


✅ 预处理器初始化完成
📊 配置信息:
  - raw_corpus_path: data/raw_corpus
  - processed_corpus_path: data/processed_corpus
  - instruction_datasets_path: data/instruction_datasets
  - evaluation_benchmark_path: data/evaluation_benchmark
  - chunk_size: 512
  - chunk_overlap: 50
  - min_chunk_length: 100
  - convert_traditional: True
  - chinese_only: True
  - min_chinese_ratio: 0.6
  - chunk_simhash_threshold: 3


## 3. Clean old files (optional)

**Important**: If you changed encoding logic or other processing logic, delete old outputs and start fresh instead of resuming from checkpoint.

**When to clean**:
- Encoding logic was updated (e.g. TXT encoding fix)
- Text cleaning rules were updated
- Deduplication logic was updated
- You want a full reprocess of all files

**When to resume**:
- Processing was only interrupted; no code changes
- You want to continue from the last checkpoint


In [3]:
# # 3. Clean old files (optional)
# # Uncomment to remove old outputs and start from scratch.

# CLEAN_OLD_FILES = True  # True = clean; False = keep

# if CLEAN_OLD_FILES:
#     processed_dir = project_root / "data" / "processed_corpus"
#     files_to_clean = [
#         processed_dir / "chunks_clean.jsonl",
#         processed_dir / "checkpoint.json",
#         processed_dir / "audit_log.jsonl",
#     ]
#     cleaned_count = 0
#     for file_path in files_to_clean:
#         if file_path.exists():
#             file_path.unlink()
#             cleaned_count += 1
#             print(f"Deleted: {file_path.name}")
#     if cleaned_count > 0:
#         print(f"Cleaned {cleaned_count} files. Next run will process all files from scratch.")
#     else:
#         print("No files to clean.")
# else:
#     print("Skipping clean; will use existing files (resume from checkpoint if present).")


In [4]:
# # Run preprocessing (produces chunks_clean.jsonl in one go).
# # Cleaning / quality / dedup are done during parse+chunk.

# # Uncomment to run (checkpoint mode recommended):
# # num_workers: CPU cores (e.g. 4, 8) for speed; None or 1 for debug.
# stats = preprocessor.process_all_files_with_checkpoint(
#     show_progress=True,
#     enable_audit=True,
#     num_workers=20,
#     batch_size=50,
#     checkpoint_interval=5
# )
# print("\n" + "="*60)
# print("Preprocessing summary")
# print("="*60)
# print(f"Total files: {stats['total_files']}")
# print(f"Successful: {stats['successful']}")
# print(f"Skipped: {stats['skipped']}")
# print(f"Total chunks: {stats['total_chunks']:,}")
# print(f"Chunks dedup: {stats.get('chunks_dedup_count', 0):,}")
# print("Output: data/processed_corpus/chunks_clean.jsonl")
# if 'audit_log_path' in stats:
#     print(f"Audit log: {stats['audit_log_path']}")


In [5]:
# # Alternative: run without checkpoint (single full pass).
# stats = preprocessor.process_all_files_clean(
#     show_progress=True,
#     enable_audit=True,
#     num_workers=multiprocessing.cpu_count()
# )
# print("\n" + "="*60)
# print("Preprocessing summary")
# print("="*60)
# print(f"Total files: {stats['total_files']}")
# print(f"Successful: {stats['successful']}")
# print(f"Skipped: {stats['skipped']}")
# print(f"Total chunks: {stats['total_chunks']:,}")
# print(f"Chunks dedup: {stats.get('chunks_dedup_count', 0):,}")
# print("Output: data/processed_corpus/chunks_clean.jsonl")
# if 'audit_log_path' in stats:
#     print(f"Audit log: {stats['audit_log_path']}")


## 4. Sample preview and quality check


In [6]:
# Sample preview and quality check
import json
import random
from collections import Counter
from src.data_processing.data_quality_check_01_04 import DataQualityChecker

checker = DataQualityChecker()
chunks_file = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"

if not chunks_file.exists():
    print("chunks_clean.jsonl not found; run preprocessing first.")
else:
    k = 5
    samples = []
    seen_valid = 0
    with open(chunks_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            ok, _ = checker.is_valid_text(obj.get("text", ""))
            if not ok:
                continue
            seen_valid += 1
            if len(samples) < k:
                samples.append(obj)
            else:
                j = random.randint(1, seen_valid)
                if j <= k:
                    samples[j - 1] = obj
    print("=" * 60)
    print("Random sample preview (quality-checked)")
    print("=" * 60)
    for i, chunk in enumerate(samples, 1):
        text = chunk.get("text", "")
        print(f"\nSample {i}:")
        print(f"  doc_id: {str(chunk.get('doc_id', 'N/A'))[:80]}...")
        print(f"  length: {len(text)} chars")
        print(f"  preview: {text[:200]}...")
    audit_log_file = project_root / "data" / "processed_corpus" / "audit_log.jsonl"
    if audit_log_file.exists():
        print("\n" + "=" * 60)
        print("Audit log stats")
        print("=" * 60)
        action_counts = Counter()
        total_audit_entries = 0
        with open(audit_log_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    entry = json.loads(line)
                    total_audit_entries += 1
                    action = entry.get("action", "unknown")
                    action_counts[action] += 1
                except Exception:
                    continue
        print(f"Total audit entries: {total_audit_entries:,}")
        print("By action:")
        for action, count in action_counts.most_common():
            print(f"  {action}: {count:,}")



📋 随机样本预览（已通过质量检查）

样本 1:
  doc_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2024_601360_2024_#ESG_三六零_三六零安全科技股份有限公...
  文本长度: 489 字符
  文本预览: 化战略为自身使命报告期内，公司结合自身能力，确立了“AI+安全”双主线发展战略，以“大模型赋能产业数字化”推动数字化向智能化升级，以“安全即服务”的理念守住数字化经济底线公司以AI重塑PC互联网业务和数字安全业务，自研千亿参数的认知型通用大模型“360智脑”，发布了企业级AI大模型解决方案，打造企业级垂直大模型，助力产业智能化转型和数字化升级目前，360大模型行业解决方案已支撑完成近20个行业垂直...

样本 2:
  doc_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2009_002175_2009_#CSR_广陆数测_2009年度社会责任报...
  文本长度: 481 字符
  文本预览: ，将员工成长、成才作为公司发展的基础公司做好员工关系维护，降低核心员工流失和用工风险在经济危机和《劳动合同法》实施的双重作用下，公司在人力资源方面谨慎处理员工离职、调岗、调薪等方面工作同时要求各部门负责人也应提高劳动法律意识，注意与员工沟通的方式与技巧在用人部门及人力资源部双方的紧密配合下，做好员工关系维护工作，尽量减少，甚至避免公司需要承担的法律风险、补偿成本和用工风险（二）教育培训方面2009...

样本 3:
  doc_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2015_600900_2015_#CSR_长江电力_2015年度社会责任报...
  文本长度: 509 字符
  文本预览: 效果图8.4开展志愿活动公司深入推进青年志愿服务工作，以关注青年、关爱青年、服务青年为重点，以“青春奉献一流”、“志愿者集中服务月”、爱心捐款为契机，深化“青春与梦想”行动，深入开展系列志愿服务和企地共建活动，贡献青春力量2015 年共组织“责任三峡、青春随行”志愿服务活动 80 余次，近 3000 人次青年员工参与案例 22：开展 2015 年度青年志愿者集中服务月活动2015 年 3 月，公司..

In [7]:
# doc_id coverage: raw_corpus (expected) vs chunks_clean.jsonl (actual)
import json
from collections import Counter
from tqdm.auto import tqdm

raw_dir = project_root / "data" / "raw_corpus"
chunks_clean = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"
if not raw_dir.exists():
    raise FileNotFoundError(f"Raw corpus dir not found: {raw_dir}")
if not chunks_clean.exists():
    raise FileNotFoundError(f"chunks_clean not found: {chunks_clean}")

raw_files = preprocessor.get_all_files(raw_dir)
expected = set(preprocessor.generate_doc_id(p, raw_dir) for p in raw_files)
actual = set()
with open(chunks_clean, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading doc_id from chunks_clean", unit="lines"):
        try:
            obj = json.loads(line)
        except Exception:
            continue
        doc_id = obj.get("doc_id")
        if doc_id:
            actual.add(doc_id)

missing = expected - actual
coverage = (len(actual) / len(expected) * 100) if expected else 0
print("=" * 60)
print("doc_id coverage")
print("=" * 60)
print(f"Expected (raw_corpus): {len(expected):,}")
print(f"Actual (chunks_clean): {len(actual):,}")
print(f"Coverage: {coverage:.2f}%")
print(f"Missing (in raw but no chunks in output): {len(missing):,}\n")

def layer_of(doc_id: str) -> str:
    return doc_id.split("_", 1)[0] if "_" in doc_id else "(unknown)"
layer_counts = Counter(layer_of(d) for d in missing)
if layer_counts:
    print("Missing by layer (top): " + ", ".join([f"{k}:{v:,}" for k, v in layer_counts.most_common(6)]))
if missing:
    print("Sample missing doc_id (first 10):")
    for d in list(missing)[:10]:
        print(f"  - {d[:140]}")
print("\nHigh doc_id coverage means whole docs are present; fewer chunks usually means some paragraphs/pages were cleaned or dropped.")


读取doc_id: chunks_clean: 0lines [00:00, ?lines/s]

doc_id 覆盖率对比
预期文档数（raw_corpus）: 17,721
实际文档数（chunks_clean）: 15,591
文档覆盖率: 87.98%
缺失文档数（raw有，但chunks_clean一个chunk都没了）: 2,130

缺失按层级(Top): 第二层:2,053, 第四层:39, 第一层:25, 第三层:13
缺失doc_id样例(前10个):
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2016_601168_2016_#CSR_西部矿业_2016年度社会责任报告_2017-03-17
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2010_600557_2010_#CSR_康缘药业_2010年度社会责任报告_2011-04-14
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2021_300003_2021_#CSR_乐普医疗_2021年社会责任报告_2022-04-25
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2017_600886_2017_#CSR_国投电力_2017年度社会责任报告_2018-03-26
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2017_600495_2017_#CSR_晋西车轴_2017年度社会责任报告_2018-03-28
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2023_002032_2023_#ESG_苏泊尔_2023年度环境、社会及治理（ESG）报告_2024-03-30
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2021_603699_2021_#CSR_纽威股份_纽威股份2021年社会责任报告_2022-04-15
  - 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2017_300095_2017_#CSR_华伍股份_2017年社会责任报告_2018-04-19
  - 第二层_社会责任报告（2006年-2024年

## 5. Chunk sample preview

Random sample of chunks (char count and content) to inspect data quality.


In [8]:
# 5. Chunk sample preview (char count and content)
import json
import random
from tqdm import tqdm

chunks_file = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"
if not chunks_file.exists():
    print(f"{chunks_file} not found; run preprocessing first.")
else:
    print("="*60)
    print("Chunk sample preview")
    print("="*60)
    chunks = []
    print("Reading chunks...")
    with open(chunks_file, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Reading chunks"):
            try:
                chunk = json.loads(line)
                chunks.append(chunk)
            except Exception:
                continue
    print(f"Read {len(chunks):,} chunks.")
    sample_count = 10
    samples = chunks if len(chunks) < sample_count else random.sample(chunks, sample_count)
    print(f"\nRandom sample of {len(samples)} chunks:\n")
    for i, chunk in enumerate(samples, 1):
        text = chunk.get("text", "")
        char_count = len(text)
        doc_id = chunk.get("doc_id", "N/A")
        chunk_id = chunk.get("chunk_id", "N/A")
        estimated_tokens = int(char_count * 1.5)
        print("="*60)
        print(f"Sample {i}/{len(samples)}")
        print("="*60)
        print(f"doc_id: {doc_id[:100]}{'...' if len(doc_id) > 100 else ''}")
        print(f"chunk_id: {chunk_id[:100]}{'...' if len(chunk_id) > 100 else ''}")
        print(f"chars: {char_count:,}")
        print(f"est. tokens: {estimated_tokens:,} (chars × 1.5)")
        print(f"text: {text[:300]}{'...' if len(text) > 300 else ''}")
        print()
    import numpy as np
    char_counts = [len(chunk.get("text", "")) for chunk in chunks]
    print("="*60)
    print("All chunks stats")
    print("="*60)
    print(f"Total chunks: {len(chunks):,}")
    print(f"Chars: mean {np.mean(char_counts):.1f}, median {np.median(char_counts):.1f}, min {np.min(char_counts)}, max {np.max(char_counts):,}, std {np.std(char_counts):.1f}")
    print("Char count distribution:")
    ranges = [(0,200,"0-200"),(200,400,"200-400"),(400,512,"400-512"),(512,800,"512-800"),(800,1200,"800-1200"),(1200,float('inf'),">1200")]
    for min_val, max_val, label in ranges:
        count = sum(1 for x in char_counts if min_val <= x < max_val)
        pct = count / len(char_counts) * 100
        print(f"  {label:12s}: {count:6,} ({pct:5.1f}%)")
    print("="*60)


📊 Chunk抽样预览
📖 正在读取chunks文件...


读取chunks: 543688it [00:03, 152379.08it/s]



✅ 共读取 543,688 个chunks

📋 随机抽样 10 个chunks预览：

样本 1/10
📄 doc_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2014_600798_2014_#CSR_宁波海运_2014年度社会责任报告_2015-03-27
🔖 chunk_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2014_600798_2014_#CSR_宁波海运_2014年度社会责任报告_2015-03-27_chunk_3
📏 字符数: 504
🔢 估算Token数: 756 (字符数 × 1.5)
📝 文本内容:
   信规范的公司治理责任体系公司一贯遵守法律法规、商业道德和公司内部控制制度，通过不断完善公司治理架构，优化管理控制体系，充分保障股东权益，不断提升公司价值，构建企业可持续发展的坚实保障（一）致力于建设稳健的公司治理结构公司不断完善 “三会一层”治理机制，通过“三会一层”的科学决策和有效运转，公司战略目标与发展定位明确，风险管控能力提升2014年，根据有关法律法规和监管部门规范运作的要求，公司不断完善制度建设，制定管理制度5项，修改1项，修订4项年内，共召开2次股东大会会议，5次董事会会议和4次监事会会议，会议的召集、召开和表决程序规范有效公司控股股东严格规范自己的行为，实际控制人及本公司高度重视...

样本 2/10
📄 doc_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2024_603299_2024_#ESG_苏盐井神_江苏苏盐井神股份有限公司2024年度环境、社会和公司治理（ES...
🔖 chunk_id: 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2024_603299_2024_#ESG_苏盐井神_江苏苏盐井神股份有限公司2024年度环境、社会和公司治理（ES...
📏 字符数: 511
🔢 估算Token数: 766 (字符数 × 1.5)
📝 文本内容:
   显著，如瑞丰盐业公司降低盐产品硫酸根含量获 QC 成果一等奖，众多员工在创新中提升技能，实现自我价值在思想文化建设上，公司计划开展职工道德大讲堂活动，从 2025 年起定期举

## 6. Token length check (full)

Check token length distribution for **all** chunks to see if reprocessing is needed.

- With `transformers`: Qwen tokenizer for exact counts
- Without: character-based estimate (~1–2 tokens per character)
- Model limit: 2048 tokens
- This run checks all chunks (no sampling).


In [9]:
# 6. Token length check (full)
import json
import numpy as np
from tqdm import tqdm

chunks_file = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"
if not chunks_file.exists():
    print(f"{chunks_file} not found; run preprocessing first.")
else:
    print("="*60)
    print("Token length distribution")
    print("="*60)
    use_tokenizer = False
    tokenizer = None
    import sys
    for m in list(sys.modules.keys()):
        if m.startswith('transformers'):
            del sys.modules[m]
    try:
        import transformers
        print(f"transformers {transformers.__version__}")
        from transformers import AutoTokenizer
        print("Loading Qwen tokenizer...")
        try:
            tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B-Chat", trust_remote_code=True)
            use_tokenizer = True
            print("Using exact token counts.")
        except Exception as e:
            print(f"Tokenizer load failed: {e}; using char estimate (~1.5 tokens/char).")
    except ImportError as e:
        print(f"transformers not found: {e}. Using char estimate. pip install transformers for exact counts.")
    token_lengths = []
    max_tokens = 0
    over_2048_count = 0
    over_1024_count = 0
    problem_chunks = []
    sample_size = 0
    with open(chunks_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
    with open(chunks_file, "r", encoding="utf-8") as f:
        for i, line in enumerate(tqdm(f, total=total_lines, desc="Analyzing chunks")):
            if sample_size > 0 and i >= sample_size:
                break
            try:
                chunk = json.loads(line)
                text = chunk.get("text", "")
                char_len = len(text)
                if use_tokenizer and tokenizer:
                    token_count = len(tokenizer.encode(text, add_special_tokens=False))
                else:
                    token_count = int(char_len * 1.5)
                token_lengths.append(token_count)
                max_tokens = max(max_tokens, token_count)
                if token_count > 2048:
                    over_2048_count += 1
                    problem_chunks.append({"chunk_id": chunk.get("chunk_id","unknown"), "tokens": token_count, "chars": char_len})
                elif token_count > 1024:
                    over_1024_count += 1
            except Exception as e:
                continue
    if not token_lengths:
        print("No valid chunks.")
    else:
        print(f"Checked {len(token_lengths):,} chunks.")
        print(f"Token stats: mean {np.mean(token_lengths):.1f}, median {np.median(token_lengths):.1f}, min {np.min(token_lengths)}, max {max_tokens}, std {np.std(token_lengths):.1f}")
        for p in [50,75,90,95,99]:
            print(f"  {p}%ile: {np.percentile(token_lengths, p):.1f} tokens")
        print("Ranges:")
        for min_v, max_v, label in [(0,256,"0-256"),(256,512,"256-512"),(512,1024,"512-1024"),(1024,2048,"1024-2048"),(2048,float('inf'),">2048")]:
            c = sum(1 for x in token_lengths if min_v <= x < max_v)
            print(f"  {label}: {c:,} ({c/len(token_lengths)*100:.1f}%)")
        print(f"Over 1024: {over_1024_count:,}; over 2048: {over_2048_count:,}")
        if over_2048_count > 0:
            print(f"First 5 over 2048:")
            for i, pc in enumerate(problem_chunks[:5], 1):
                print(f"  {i}. {pc['chunk_id'][:80]}...: {pc['tokens']} tokens")
        else:
            print("All chunks within 2048 tokens.")
        print("="*60)
        if over_2048_count == 0:
            print("No reprocessing needed.")
        elif over_2048_count < 10:
            print("Few over limit; truncate at train time or fix manually.")
        else:
            print("Many over limit; check chunking and consider reprocessing.")
        if not use_tokenizer:
            print("pip install transformers for exact token stats.")
        print("="*60)


📊 Token长度分布检查
✅ 检测到transformers库，版本: 4.57.3


2025-12-19 10:45:08,788 - numexpr.utils - INFO - Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2025-12-19 10:45:08,788 - numexpr.utils - INFO - NumExpr defaulting to 16 threads.


📥 正在加载Qwen tokenizer（首次加载需要下载模型文件，可能需要几分钟）...


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\江亦晗\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\江亦晗\.cache\huggingface\hub\models--Qwen--Qwen1.5-7B-Chat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer加载成功，将使用精确Token统计

📖 正在读取chunks文件...


分析chunks: 100%|██████████| 543688/543688 [04:40<00:00, 1937.60it/s]



📈 统计结果（检查了 543,688 个chunks）
✅ 已检查全部 543,688 个chunks

📊 Token长度统计:
  平均Token数: 277.8
  中位数Token数: 287.0
  最小Token数: 43
  最大Token数: 2350
  标准差: 59.1

📊 分位数统计:
  50%分位数: 287.0 tokens
  75%分位数: 308.0 tokens
  90%分位数: 328.0 tokens
  95%分位数: 342.0 tokens
  99%分位数: 373.0 tokens

📊 Token长度分布区间:
  0-256       : 94,415 ( 17.4%)
  256-512     : 448,551 ( 82.5%)
  512-1024    :    650 (  0.1%)
  1024-2048   :     70 (  0.0%)
  >2048       :      2 (  0.0%)

⚠️ 问题chunks统计:
  超过1024 tokens: 69 (0.0%)
  超过2048 tokens: 2 (0.0%)

❌ 发现 2 个chunks超过2048 tokens限制！
前5个问题chunks:
  1. 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2022_603833_2022_#ESG_欧派家居_2022年度ESG报告...: 2182 tokens (2999 chars)
  2. 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2022_688248_2022_#ESG_南网科技_南网科技：2022年环...: 2350 tokens (3194 chars)

💡 建议
⚠️ 有少量chunks超过限制，但影响不大
   - 可以考虑在训练时自动截断
   - 或者手动修复这几个chunks
   - 不需要全部重新处理


## 7. Fix chunks over token limit

Truncate chunks over 2048 tokens to within the limit.


In [10]:
# 7. Fix chunks over token limit
import json
import sys
from pathlib import Path
from tqdm import tqdm

if 'transformers' in sys.modules:
    for m in list(sys.modules.keys()):
        if m.startswith('transformers'):
            del sys.modules[m]

chunks_file = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"
output_file = project_root / "data" / "processed_corpus" / "chunks_clean_fixed.jsonl"
max_tokens = 2048

if not chunks_file.exists():
    print(f"{chunks_file} not found; run preprocessing first.")
else:
    print("="*60)
    print("Fix chunks over token limit")
    print("="*60)
    tokenizer = None
    try:
        from transformers import AutoTokenizer
        print("Loading Qwen tokenizer...")
        tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-7B-Chat", trust_remote_code=True)
        print("Tokenizer loaded.")
    except Exception as e:
        print(f"Tokenizer failed: {e}; using char estimate.")
    def find_sentence_boundary(text: str, max_chars: int) -> int:
        sentence_endings = ['。', '！', '？', '.', '!', '?', '\n']
        for i in range(max_chars, max(0, max_chars - 200), -1):
            if i < len(text) and text[i] in sentence_endings:
                return i + 1
        for i in range(max_chars, min(len(text), max_chars + 200)):
            if i < len(text) and text[i] in sentence_endings:
                return i + 1
        return max_chars
    def truncate_chunk_by_tokens(text: str, tokenizer, max_tokens: int = 2048) -> str:
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) <= max_tokens:
            return text
        truncated_tokens = tokens[:max_tokens]
        truncated_text = tokenizer.decode(truncated_tokens, skip_special_tokens=True)
        estimated_chars = int(max_tokens * 1.3)
        if estimated_chars < len(text):
            boundary = find_sentence_boundary(text, estimated_chars)
            truncated_text = text[:boundary]
            if len(tokenizer.encode(truncated_text, add_special_tokens=False)) > max_tokens:
                truncated_text = tokenizer.decode(truncated_tokens, skip_special_tokens=True)
        return truncated_text
    total_chunks = 0
    fixed_chunks = 0
    oversized_chunks = []
    with open(chunks_file, "r", encoding="utf-8") as f:
        total_lines = sum(1 for _ in f)
    print(f"Total chunks: {total_lines:,}\n")
    with open(chunks_file, "r", encoding="utf-8") as f_in, open(output_file, "w", encoding="utf-8") as f_out:
        for line in tqdm(f_in, total=total_lines, desc="Processing"):
            try:
                chunk = json.loads(line)
                text = chunk.get("text", "")
                total_chunks += 1
                if tokenizer:
                    token_count = len(tokenizer.encode(text, add_special_tokens=False))
                else:
                    token_count = int(len(text) * 1.5)
                if token_count > max_tokens:
                    oversized_chunks.append({"chunk_id": chunk.get("chunk_id","unknown"), "original_tokens": token_count, "original_chars": len(text)})
                    if tokenizer:
                        fixed_text = truncate_chunk_by_tokens(text, tokenizer, max_tokens)
                    else:
                        boundary = find_sentence_boundary(text, int(max_tokens / 1.5))
                        fixed_text = text[:boundary]
                    chunk["text"] = fixed_text
                    fixed_chunks += 1
                f_out.write(json.dumps(chunk, ensure_ascii=False) + "\n")
            except Exception:
                continue
    print("\n" + "="*60)
    print("Fix summary")
    print("="*60)
    print(f"Total: {total_chunks:,}; fixed: {fixed_chunks:,}")
    if total_chunks > 0:
        print(f"Fixed %: {fixed_chunks/total_chunks*100:.2f}%")
    for i, oc in enumerate(oversized_chunks[:10], 1):
        print(f"  {i}. {oc['chunk_id'][:80]}... (was {oc['original_tokens']} tokens)")
    print(f"Input: {chunks_file}")
    print(f"Output: {output_file}")
    if fixed_chunks > 0:
        print("Check output; if OK, replace original or run cell 8.")
    print("="*60)


🔧 修复超过Token限制的Chunks
📥 正在加载tokenizer: Qwen/Qwen1.5-7B-Chat
✅ Tokenizer加载成功

📖 正在读取chunks文件...
📊 总共 543,688 个chunks



处理chunks: 100%|██████████| 543688/543688 [04:42<00:00, 1923.54it/s]


📊 修复统计
总chunks数: 543,688
修复的chunks数: 2
修复比例: 0.00%

修复的chunks详情:
  1. 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2022_603833_2022_#ESG_欧派家居_2022年度ESG报告...
     原始: 2182 tokens (2999 chars)
  2. 第二层_社会责任报告（2006年-2024年）_社会责任报告（去重）_17082份_2022_688248_2022_#ESG_南网科技_南网科技：2022年环...
     原始: 2350 tokens (3194 chars)

✅ 修复完成！
   输入文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean.jsonl
   输出文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean_fixed.jsonl

💡 建议:
   1. 检查修复后的文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean_fixed.jsonl
   2. 如果满意，可以替换原文件:
      chunks_clean.jsonl -> chunks_clean_fixed.jsonl
   3. 或者手动验证修复后的chunks质量


## 8. Replace original file (optional)

If the fixed file looks good, replace the original with it.


In [13]:
# 8. Replace original file (optional)
import shutil

chunks_file = project_root / "data" / "processed_corpus" / "chunks_clean.jsonl"
fixed_file = project_root / "data" / "processed_corpus" / "chunks_clean_fixed.jsonl"
backup_file = project_root / "data" / "processed_corpus" / "chunks_clean_backup.jsonl"
REPLACE_ORIGINAL = True

if not fixed_file.exists():
    print(f"Fixed file not found: {fixed_file}. Run cell 7 first.")
elif not REPLACE_ORIGINAL:
    print("Replace disabled. Set REPLACE_ORIGINAL = True to replace (original will be backed up).")
    print(f"Fixed: {fixed_file}\nOriginal: {chunks_file}")
else:
    print("="*60)
    print("Replacing original file")
    print("="*60)
    if chunks_file.exists():
        print("Backing up original...")
        shutil.copy2(chunks_file, backup_file)
        print(f"Backup: {backup_file}")
    print("Replacing...")
    shutil.copy2(fixed_file, chunks_file)
    print("Done.")
    print(f"Current: {chunks_file}\nBackup: {backup_file}\nFixed: {fixed_file}")
    print("="*60)


🔄 替换原文件
📦 正在备份原文件...
✅ 已备份到: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean_backup.jsonl
🔄 正在替换原文件...
✅ 替换完成！

📁 文件状态:
   当前文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean.jsonl
   备份文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean_backup.jsonl
   修复文件: D:\9_Projects\Veritas\VeritasCarbon_VLDB\data\processed_corpus\chunks_clean_fixed.jsonl
